---
sidebar_label: TensorFeed
---

# TensorFeedLoader

>[TensorFeed.ai](https://tensorfeed.ai) aggregates AI news from 12+ industry sources (Anthropic, OpenAI, Google AI, Meta AI, HuggingFace, NVIDIA, TechCrunch, The Verge, Hacker News, and more).

`TensorFeedLoader` streams those articles in as LangChain `Document` objects so you can index them into a vector store for RAG over current AI industry coverage. The loader uses the free public TensorFeed API and requires no API key.

## Overview

### Integration details

| Class | Package | Local | Lazy loading |
| --- | --- | --- | --- |
| [TensorFeedLoader](https://github.com/RipperMercs/tensorfeed/blob/main/sdk/langchain-python/langchain_tensorfeed/loaders.py) | `langchain-tensorfeed` | No | Yes |

### Loader features

- API-side category filter (single category)
- Client-side multi-category filter
- Inclusive `start_date` / `end_date` filtering on `publishedAt`
- Lazy iterator support via `lazy_load()`

## Setup

No credentials are required.

### Installation

In [ ]:
%pip install --quiet --upgrade langchain-tensorfeed

## Initialization

Construct the loader with the filters you need.

In [ ]:
from langchain_tensorfeed import TensorFeedLoader

loader = TensorFeedLoader(
    category="research",
    limit=100,
    start_date="2026-04-01T00:00:00Z",
)

## Load

`load()` returns a list of `Document` objects with `page_content` set to the article title plus snippet, and structured metadata on each document.

In [ ]:
docs = loader.load()
print(f"Loaded {len(docs)} documents")
print(docs[0].page_content[:200])

In [ ]:
docs[0].metadata

Metadata fields:

- `id` — TensorFeed article id
- `title` — article headline
- `source` — publisher (e.g. `Anthropic Blog`, `Hacker News AI`)
- `source_domain` — root domain of the original article
- `url` — original article URL
- `categories` — list of category labels
- `published_at` — ISO 8601 UTC timestamp
- `fetched_at` — when TensorFeed pulled the article from its source

## Lazy load

Stream documents one at a time. Useful when piping into a text splitter or vector store with backpressure.

In [ ]:
first_three = []
for doc in TensorFeedLoader(limit=50).lazy_load():
    first_three.append(doc)
    if len(first_three) == 3:
        break
first_three

## Filtering by multiple categories

Pass `categories` for client-side filtering after the API call. An article is kept if any of its categories match (case-insensitive).

In [ ]:
loader = TensorFeedLoader(categories=["anthropic", "openai"], limit=100)
docs = loader.load()
print(f"{len(docs)} documents from Anthropic or OpenAI")

## Use in a RAG pipeline

Split the loaded documents and embed them into a vector store, then query with any LangChain retriever.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_tensorfeed import TensorFeedLoader

raw_docs = TensorFeedLoader(limit=200).load()
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
splits = splitter.split_documents(raw_docs)

vectorstore = FAISS.from_documents(splits, OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

results = retriever.invoke("What did Anthropic announce this week?")
for r in results:
    print(r.metadata["title"], "->", r.metadata["url"])

## API reference

- Source code: [github.com/RipperMercs/tensorfeed](https://github.com/RipperMercs/tensorfeed) under `sdk/langchain-python/`
- TensorFeed developer docs: [tensorfeed.ai/developers](https://tensorfeed.ai/developers)
- News API endpoint: [tensorfeed.ai/api/news](https://tensorfeed.ai/api/news)